# OpenDDE on Colab

Requires a GPU runtime: **Runtime > Change runtime type > T4 GPU** (or better).

This notebook: installs OpenDDE from source with `uv`, downloads model weights + inference data, runs a tiny smoke test, then runs your real prediction job.

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!test -d OpenDDE || git clone https://github.com/aurekaresearch/OpenDDE.git
%cd OpenDDE
!uv pip install --torch-backend cu126 -e '.[gpu]'
!uv pip install --group dev
!opendde doctor

In [ ]:
# General-purpose checkpoint used by default with -n opendde_v1.
%env OPENDDE_ROOT_DIR=/content/data
!mkdir -p $OPENDDE_ROOT_DIR/checkpoint

In [ ]:
# The download script needs the `zstd` CLI (not the Python `zstd` package) to
# decompress the search-database .zst files, or it silently falls back to a
# broken S3 mirror (403).
!apt-get -qq update && apt-get -qq install -y zstd
!bash /content/OpenDDE/scripts/download_opendde_data.sh
# If you don't need MSA/template/RNA-MSA features (as in the pred command below),
# add --skip-search-database to skip several GB of files you won't use:
# !bash /content/OpenDDE/scripts/download_opendde_data.sh --skip-search-database

In [ ]:
# Smoke test with the minimal example from the OpenDDE README, to confirm
# the install + data download actually work before running a real job.
# --need_atom_confidence true is required to get a *_full_data_sample_*.json
# per model (PAE matrix, per-atom pLDDT) - needed for the PAE viewer and ipSAE below.
tiny_json = '''[
    {
        "name": "tiny",
        "modelSeeds": [101],
        "sequences": [
            {
                "proteinChain": {
                    "sequence": "ACDEFGHIK",
                    "count": 1
                }
            }
        ]
    }
]'''
with open("tiny.json", "w") as f:
    f.write(tiny_json)

!opendde pred -i tiny.json -o ./output -n opendde_v1 --use_msa false --use_template false --use_rna_msa false --sample 1 --step 200 --cycle 10 --need_atom_confidence true

In [ ]:
# Your real job. Upload/create egfr-rad51-kinase.json in this working directory
# first (it wasn't part of the original notebook, so it must exist before running).
!opendde pred -i egfr-rad51-kinase.json -o ./output -n opendde_v1 --use_msa false --use_template false --use_rna_msa false --sample 1 --step 200 --cycle 10 --need_atom_confidence true

## Post-processing: visualize structures + rank by ipSAE

Requires the `pred` runs above to have used `--need_atom_confidence true` (adds `*_full_data_sample_*.json` per model, with the PAE matrix and per-atom pLDDT).

In [ ]:
# Shared helpers: locate each model's .cif plus its matching per-sample JSON
# sidecars, following OpenDDE's dumper.py naming convention:
#   {name}_sample_{rank}.cif
#   {name}_summary_confidence_sample_{rank}.json   (always written)
#   {name}_full_data_sample_{rank}.json             (only with --need_atom_confidence true)
import glob
import os
import re

_SAMPLE_RE = re.compile(r"(.+)_sample_(\d+)\.cif$")


def find_all_cifs(output_dir="./output"):
    return sorted(glob.glob(os.path.join(output_dir, "**", "*.cif"), recursive=True))


def _sample_stem_rank(cif_path):
    m = _SAMPLE_RE.match(os.path.basename(cif_path))
    return m.groups() if m else (None, None)


def find_full_data_json(cif_path):
    stem, rank = _sample_stem_rank(cif_path)
    if stem is None:
        return None
    p = os.path.join(os.path.dirname(cif_path), f"{stem}_full_data_sample_{rank}.json")
    return p if os.path.exists(p) else None


def find_summary_json(cif_path):
    stem, rank = _sample_stem_rank(cif_path)
    if stem is None:
        return None
    p = os.path.join(os.path.dirname(cif_path), f"{stem}_summary_confidence_sample_{rank}.json")
    return p if os.path.exists(p) else None


cifs = find_all_cifs()
print(f"Found {len(cifs)} predicted structure(s) under ./output")

In [ ]:
#@title Visualize predicted structures + PAE (py2Dmol)
!pip install -q py3Dmol py2Dmol

import json

import numpy as np
import py2Dmol

load_as_frames = True  # @param {type:"boolean"}
viewer_size = 400


def load_pae(cif_path):
    # OpenDDE's full_data.json stores the PAE matrix as "token_pair_pae"
    # (AlphaFold3's equivalent full_data.json key is "pae").
    fd_path = find_full_data_json(cif_path)
    if fd_path is None:
        return None
    with open(fd_path) as f:
        data = json.load(f)
    pae = data.get("token_pair_pae")
    return np.asarray(pae, dtype=float) if pae is not None else None


if not cifs:
    print("No .cif files found under ./output yet - run a prediction first.")
else:
    viewer = py2Dmol.view(size=(viewer_size, viewer_size), pae=True, autoplay=load_as_frames)
    for i, cif in enumerate(cifs, start=1):
        pae = load_pae(cif)
        if pae is None:
            print(f"No PAE data for {cif} (re-run pred with --need_atom_confidence true)")
        name = "models" if load_as_frames else os.path.basename(cif).replace(".cif", "")
        viewer.add_pdb(cif, name=name, paes=pae)
    viewer.show()

In [ ]:
#@title Compute ipSAE for every model and rank by decreasing ipSAE
# Uses the Dunbrack Lab's ipsae.py (https://github.com/DunbrackLab/IPSAE),
# which reads AlphaFold3-style keys ("pae", "atom_plddts") from a
# "*_full_data*.json" file and an optional sibling "*_summary_confidences*.json"
# (for "chain_pair_iptm", used to populate the ipTM_af column). OpenDDE's
# full_data.json uses different key names ("token_pair_pae", "atom_plddt") and
# a different sidecar naming convention, so we adapt/copy each sample's JSON
# into ipsae.py's expected shape before invoking it.
import glob
import json
import os
import subprocess
import sys

import pandas as pd

!curl -fsSL -o ipsae.py https://raw.githubusercontent.com/DunbrackLab/IPSAE/main/ipsae.py

PAE_CUTOFF = 10  # @param {type:"number"}
DIST_CUTOFF = 10  # @param {type:"number"}
IPSAE_TMP_DIR = "ipsae_tmp"
os.makedirs(IPSAE_TMP_DIR, exist_ok=True)

rows = []
for cif_path in cifs:
    full_data_path = find_full_data_json(cif_path)
    if full_data_path is None:
        print(f"Skipping {cif_path}: no *_full_data_sample_*.json "
              f"(re-run pred with --need_atom_confidence true)")
        continue

    try:
        with open(full_data_path) as f:
            full_data = json.load(f)
        if "token_pair_pae" not in full_data or "atom_plddt" not in full_data:
            print(f"Skipping {cif_path}: full_data.json missing token_pair_pae/atom_plddt")
            continue

        stem = os.path.basename(cif_path).replace(".cif", "")

        # Adapted PAE/confidence file, named so ipsae.py's "full_data" path
        # substitution can find our adapted summary file below.
        adapted_pae_path = os.path.join(IPSAE_TMP_DIR, f"{stem}_full_data.json")
        with open(adapted_pae_path, "w") as f:
            json.dump({"pae": full_data["token_pair_pae"],
                       "atom_plddts": full_data["atom_plddt"]}, f)

        summary_path = find_summary_json(cif_path)
        if summary_path is not None:
            with open(summary_path) as f:
                summary_data = json.load(f)
            if "chain_pair_iptm" in summary_data:
                adapted_summary_path = os.path.join(IPSAE_TMP_DIR, f"{stem}_summary_confidences.json")
                with open(adapted_summary_path, "w") as f:
                    json.dump({"chain_pair_iptm": summary_data["chain_pair_iptm"]}, f)

        result = subprocess.run(
            [sys.executable, "ipsae.py", adapted_pae_path, cif_path,
             str(PAE_CUTOFF), str(DIST_CUTOFF)],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            print(f"ipsae.py failed on {cif_path}:\n{result.stderr}")
            continue

        score_files = [p for p in glob.glob(cif_path.replace(".cif", "") + "*.txt")
                       if not p.endswith("_byres.txt")]
        if not score_files:
            print(f"No ipsae.py output found for {cif_path}")
            continue

        with open(score_files[0]) as f:
            lines = [l for l in f if l.strip()]
        header = lines[0].split()
        scores = pd.DataFrame([l.split() for l in lines[1:]], columns=header)
        scores["ipSAE"] = scores["ipSAE"].astype(float)
        # Every row is already an inter-chain pair (ipsae.py skips chain1==chain2);
        # take the strongest interface as this model's representative score.
        best = scores.loc[scores["ipSAE"].idxmax()]

        rows.append({
            "model": os.path.relpath(cif_path, "./output"),
            "chain_1": best["Chn1"],
            "chain_2": best["Chn2"],
            "ipSAE": float(best["ipSAE"]),
            "ipSAE_d0chn": float(best["ipSAE_d0chn"]),
            "ipTM_af": float(best["ipTM_af"]),
            "pDockQ": float(best["pDockQ"]),
        })
    except Exception as e:
        print(f"Skipping {cif_path}: {e}")

ranking_df = pd.DataFrame(rows).sort_values("ipSAE", ascending=False).reset_index(drop=True)
ranking_df.to_csv("ipsae_ranking.csv", index=False)
ranking_df

In [ ]:
from google.colab import files

files.download("ipsae_ranking.csv")